In [1]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader

loader = DirectoryLoader('docs', glob='*.pdf', loader_cls=PyPDFLoader)
documents = loader.load()
print(f"Loaded {len(documents)} pages")

Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 10 0 (offset 0)
Ignoring wrong pointing object 12 0 (offset 0)
Ignoring wrong pointing object 14 0 (offset 0)
Ignoring wrong pointing object 16 0 (offset 0)
Ignoring wrong pointing object 18 0 (offset 0)
Ignoring wrong pointing object 20 0 (offset 0)
Ignoring wrong pointing object 22 0 (offset 0)
Ignoring wrong pointing object 24 0 (offset 0)
Ignoring wrong pointing object 26 0 (offset 0)
Ignoring wrong pointing object 30 0 (offset 0)
Ignoring wrong pointing object 32 0 (offset 0)
Ignoring wrong pointing object 34 0 (offset 0)
Ignoring wrong pointing object 36 0 (offset 0)
Ignoring wrong pointing object 38 0 (offset 0)


Loaded 43 pages


In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(documents)
print(f"Created {len(chunks)} chunks")

Created 293 chunks


In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [4]:
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(documents)
print(f"Created {len(chunks)} chunks")

Created 293 chunks


In [5]:
print(chunks[0].page_content)

YOUR CA T’S
NUTR ITIONAL NEEDS
A Science-Based Guide For Pet Owners
37491_Cat_P01_16  07/24/06  4:53 PM  Page 1


In [6]:
print(chunks[10].page_content)
print("---")
print(chunks[50].page_content)

2
C
ats need several different kinds of nutrients to survive: amino acids
from protein, fatty acids and carbohydrates, vitamins, minerals, and
water.
The tables in this pamphlet provide recommended daily
allowances for nutrients based on the amount required to maintain
good health in normal cats. Your cat’s unique nutritional requirements
will depend on its size and its stage in life, among other factors. A better under-
standing of how cats use the various nutrients in food and how much of them
---
Fresh water should be available at all times,
but the amount needed varies with the type of
diet and the environmental conditions.  Cats
don’t drink as much per kilogram of body
weight as do dogs, perhaps because of their
evolution as desert animals.  Cats will drink
approximately 2 milliliters of water for every
gram of dry food they eat.  Whereas dogs will
drink enough water to replace 6% of their body
weight in one hour, cats will take 24 hours to do the


In [7]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

embeddings = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')
vectorstore = Chroma.from_documents(chunks, embeddings, persist_directory='./chroma_db')
print("Vector store created")

C:\Users\dan25\AppData\Local\Temp\ipykernel_39152\1401473188.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vector store created


In [8]:
retriever = vectorstore.as_retriever(search_kwargs={'k': 3})

query = "How much water does a cat need?"
results = retriever.invoke(query)

for i, doc in enumerate(results):
    print(f"\n--- Chunk {i+1} ---")
    print(doc.page_content)


--- Chunk 1 ---
Fresh water should be available at all times,
but the amount needed varies with the type of
diet and the environmental conditions.  Cats
don’t drink as much per kilogram of body
weight as do dogs, perhaps because of their
evolution as desert animals.  Cats will drink
approximately 2 milliliters of water for every
gram of dry food they eat.  Whereas dogs will
drink enough water to replace 6% of their body
weight in one hour, cats will take 24 hours to do the

--- Chunk 2 ---
Fresh water should be available at all times,
but the amount needed varies with the type of
diet and the environmental conditions.  Cats
don’t drink as much per kilogram of body
weight as do dogs, perhaps because of their
evolution as desert animals.  Cats will drink
approximately 2 milliliters of water for every
gram of dry food they eat.  Whereas dogs will
drink enough water to replace 6% of their body
weight in one hour, cats will take 24 hours to do the

--- Chunk 3 ---
Fresh water should be ava

In [9]:
query = "What protein do dogs need?"
results = retriever.invoke(query)

for i, doc in enumerate(results):
    print(f"\n--- Chunk {i+1} ---")
    print(doc.page_content)


--- Chunk 1 ---
they need can help you choose a healthy diet for your pet.
PROTEINS AND AMINO ACIDS
As carnivorous animals, cats derive most of their protein from meat, fish,
and other animal products.  Some animal-based protein is easier to
digest than plant-based protein and is better suited to the cat’s
digestive system. 
Dietary protein contains 10 specific amino acids that
neither cats nor dogs can make on their own.
Known as essential amino acids, they provide the

--- Chunk 2 ---
they need can help you choose a healthy diet for your pet.
PROTEINS AND AMINO ACIDS
As carnivorous animals, cats derive most of their protein from meat, fish,
and other animal products.  Some animal-based protein is easier to
digest than plant-based protein and is better suited to the cat’s
digestive system. 
Dietary protein contains 10 specific amino acids that
neither cats nor dogs can make on their own.
Known as essential amino acids, they provide the

--- Chunk 3 ---
they need can help you choose a

In [11]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOllama(model='llama3:8b', temperature=0)

def ask_question(question):
    # Step 1: Retrieve relevant chunks
    docs = retriever.invoke(question)
    
    # Step 2: Combine chunks into one context string
    context = "\n\n".join([doc.page_content for doc in docs])
    
    # Step 3: Build the prompt
    prompt = f"""Use the following context to answer the question. 
If the answer is not in the context, say "I don't have enough information to answer that."

Context:  {context}

Question: {question}

Answer:"""
    
    # Step 4: Send to LLM
    response = llm.invoke(prompt)
    return response.content

print(ask_question("How much water does a cat need daily?"))

According to the context, cats will drink approximately 2 milliliters of water for every gram of dry food they eat. Since there is no information provided about the amount of dry food a cat eats per day, I don't have enough information to answer that. However, it can be inferred that the daily water intake would depend on the amount of dry food consumed by the cat.


In [12]:
print(ask_question("What protein do cats need in their diet?"))
print("---")
print(ask_question("Is raw food safe for dogs?"))
print("---")
print(ask_question("What is the best smartphone to buy?"))

According to the context, cats derive most of their protein from meat, fish, and other animal products. Additionally, dietary protein contains 10 specific amino acids that neither cats nor dogs can make on their own, known as essential amino acids. Therefore, the answer is:

Meat, fish, and other animal-based proteins.

Note: The context does not specifically mention a particular type of protein that cats need in their diet, but rather emphasizes the importance of animal-based sources of protein for feline nutrition.
---
According to the context, there is growing evidence that feeding raw meat can be a health risk both for the dog and its owner. Additionally, raw meat-based diets have a high risk of bacterial contamination. Therefore, based on this information, it appears that raw food may not be safe for dogs.
---
I don't have enough information to answer that. The provided context only discusses pet owners, veterinarians, healthcare options, and decision-making processes, but does no

In [13]:
while True:
    question = input("\nAsk a question about pet nutrition (or 'quit'): ")
    if question.lower() == 'quit':
        break
    print(f"\nAnswer: {ask_question(question)}")


Ask a question about pet nutrition (or 'quit'):  What food do cats eat?



Answer: According to the context, cats are descended from strict meat-eaters and their behavior reveals their carnivorous nature. They seek small prey such as mice, birds, and insects when hunting. They may even kill and eat a rabbit. Therefore, the answer is:

Meat and/or fish products.



Ask a question about pet nutrition (or 'quit'):  Are cats more aggressive than dogs?



Answer: I don't have enough information to answer that. The provided context only discusses the eating habits and preferences of cats compared to dogs, but does not mention anything about aggression or temperament.



Ask a question about pet nutrition (or 'quit'):  Why are cats gay?



Answer: I don't have enough information to answer that. The provided context is a science-based guide for pet owners about cat nutrition, but it doesn't mention anything about the sexual orientation of cats or why they might be "gay."



Ask a question about pet nutrition (or 'quit'):  What food eat by cat compared to dog



Answer: According to the context, cats will not eat a powdered, commercial diet, whereas dogs may eat such a diet. Additionally, cats prefer their food in gel form, while dogs do not have this preference mentioned.



Ask a question about pet nutrition (or 'quit'):  quit
